In [1]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
import torch

torch_dtype = torch.bfloat16
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"

initial_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch_dtype,
    attn_implementation="flash_attention_2",
    device_map={"": 0},
)

from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper

cell = MemoryCell(initial_model, num_mem_tokens=16)
model = RecurrentWrapper(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
# tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
# temp = tokenizer("test")
keys = ["input_ids", "attention_mask", "labels"]
test_input_data = {}
for key in keys:
    test_input_data[key] = torch.ones(
        (1, 2048),
        device=initial_model.device,
        dtype=torch.long,
    )
test_input_data

{'input_ids': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'labels': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}

In [4]:
model.memory_cell.model.dtype

torch.bfloat16

In [5]:
model.memory_cell.memory.dtype

torch.bfloat16

In [6]:
with torch.no_grad():
    result = model(**test_input_data)

result

CausalLMOutputWithCrossAttentions(loss=tensor(3.5000, device='cuda:0', dtype=torch.bfloat16), logits=tensor([[[ 5.5312,  6.3438,  4.9688,  ..., -1.5703, -1.5703, -1.5703],
         [ 6.2188,  9.5625,  5.4375,  ..., -1.9531, -1.9531, -1.9531],
         [ 7.2500, 10.3750,  6.2812,  ..., -1.8750, -1.8750, -1.8750],
         ...,
         [ 8.3125, 13.5000,  8.5625,  ..., -1.1875, -1.1875, -1.1875],
         [ 5.8750, 10.8125,  5.0000,  ..., -1.5859, -1.5859, -1.5859],
         [ 4.7500,  8.9375,  3.9375,  ..., -2.4219, -2.4219, -2.4219]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None, cross_attentions=None)

In [7]:
result.logits

tensor([[[ 9.1875, 11.6875,  8.4375,  ...,  0.0835,  0.0835,  0.0835],
         [11.6875, 13.1875,  8.8125,  ..., -0.0449, -0.0449, -0.0449],
         [12.1250, 13.1250,  8.3750,  ..., -0.3125, -0.3125, -0.3125],
         ...,
         [ 5.1875, 10.0000,  1.1094,  ..., -2.4062, -2.4062, -2.4062],
         [ 5.1562,  9.9375,  1.0781,  ..., -2.4219, -2.4219, -2.4219],
         [ 5.1250,  9.9375,  1.2109,  ..., -2.3750, -2.3750, -2.3750]]],
       device='cuda:0', dtype=torch.bfloat16)

#### generate

In [ ]:
test_input_data["input_ids"].shape

torch.Size([1, 2048])

In [14]:
result = model.generate(
    # input_ids=test_input_data["input_ids"],
    input_ids=torch.randint(
        low=0,
        high=128,
        # size=(1, 2),
        size=(32, 77),
        dtype=torch.long,
        device="cuda",
    ),
    # attention_mask=test_input_data["attention_mask"],
    max_new_tokens=20,
)
# tokenizer.
result

tensor([[    91,     77,     91,     57,     91,     89,     91,  53498,     61,
             71,     91,     64,     91,     82,     91,     16,     91,     17,
             91,     18],
        [    93,     67,     93,     61,     93,     79,     93,     79,     93,
             67,     93,     61,     93,     84,     93,     59,     93,     93,
             70,     93],
        [    82,      5,     63,     14,     67,      5,     14,     67,     63,
             14,     67,      5,     14,     67,     63,     14,     67,     63,
             14,     67],
        [    45,     59,     61,     77,     59,     63,     62,     82,     13,
             93,     59,     63,     38,     13,     93,      0,     38,     13,
             93,      0],
        [    71,      2,     86,     51,     59,     91,     62,     73,     32,
             59,     91,     62,     73,     32,     59,     91,     62,     73,
             32,     59],
        [    91,     32,     15,     59,     61,     41,    

In [ ]:
getattr(model, "memory_cell", None) is None

False

In [ ]:
getattr(model, "memory_cell", None).memory.shape[0]

16

### Optimize train code

In [ ]:
config = {
    "k2": -1,
    "max_n_segments": 128,
    "return_all_logits": False,
    "segment_size": 1024,
    "vary_n_segments": False,
}

In [1]:
from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

model_name = "unsloth/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    # device_map={"": 0},
)
config = AutoConfig.from_pretrained(model_name)
# model = RMTForReasoning.from_pretrained(
#     model_name,
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
# )
cell = MemoryCell(
    model,
    num_mem_tokens=16,
)
model = RecurrentWrapper(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
model.load_state_dict(torch.load('./model_best.pt'))

<All keys matched successfully>

In [2]:
import torch

torch.tensor(123, device='cuda:0')

/home/user-name-goes-here/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


tensor(123, device='cuda:0')